# ResNet18 ex novo v8 - K-Fold CV leggero + Reject Option

Variante v8: K-Fold leggero per confronto in relazione, con meno epoche e TTA disattivata di default.

**Ereditato da v6.1:**
- Two-phase training (freeze breve → unfreeze)
- Focal Loss (gamma=2.0)
- Dropout 0.30, weight decay 2e-4, early stopping aggressivo
- Strong augmentation; TTA opzionale per non rallentare il K-Fold

**Novita v8:**
- **K-Fold leggero**: 15 epoche max, freeze 3 epoche, patience 4.
- **TTA reject disattivata di default**: il confronto da relazione usa K-Fold BA e reject standard.
- **Label smoothing 0.10** (da 0.15): Focal Loss gestisce gia overconfidence, LS piu basso affina boundary Plastic-Glass.
- **K-Fold Cross-Validation (K=3, StratifiedKFold)**: stima piu robusta della generalizzazione (S:04, L:01).
- **Reject Option (Vento et al., S:11)**: due funzioni di affidabilita:
  - ψ_a = max softmax confidence (detect out-of-distribution)
  - ψ_b = top-2 margin (detect ambiguita near-boundary)
  - Soglia ottimale via funzione di efficacia Ƥ con costi Ce=10, Cr=3, Cc=1 (da L:reject_option).

Obiettivo: massimizzare BA con reject option, minimizzando il reject rate.

## 1. Mount Drive e clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, datetime, subprocess
from getpass import getpass

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BASE_BRANCH = 'exp/resnet18-exnovo-v8'
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M')
RUN_BRANCH = f'exp-resnet18-exnovo-v8-{RUN_ID}'
RUN_NAME = f'resnet18_exnovo_v8_{RUN_ID}'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

def sanitize(text):
    if 'AUTH_REPO_URL' in globals():
        text = text.replace(AUTH_REPO_URL, 'https://***@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git')
    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:
        text = text.replace(GITHUB_TOKEN, '***')
    return text

def run(cmd, cwd=None, check=True):
    printable = sanitize(' '.join(cmd))
    print('$', printable)
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout: print(sanitize(result.stdout))
    if result.stderr: print(sanitize(result.stderr))
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {printable}')
    return result

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_PATH):
    run(['git', 'clone', AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente nel runtime.')
run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'fetch', 'origin', BASE_BRANCH], cwd=REPO_PATH)
run(['git', 'switch', '--detach', 'FETCH_HEAD'], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)
branch_exists = subprocess.run(['git', 'rev-parse', '--verify', RUN_BRANCH], cwd=REPO_PATH, text=True, capture_output=True).returncode == 0
if branch_exists:
    run(['git', 'switch', RUN_BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'switch', '-c', RUN_BRANCH], cwd=REPO_PATH)
os.chdir(REPO_PATH)
print('Run branch:', RUN_BRANCH)

## 2. Configurazione

In [ ]:
import json, random, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, Subset
from torchvision import models, transforms
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

EXP_NAME = 'resnet18_exnovo_v8'
SEED = 42
NUM_CLASSES = 8
BATCH_SIZE = 32
NUM_WORKERS = 0
K_FOLDS = 3
EPOCHS = 15
PATIENCE = 4
TWO_PHASE_FREEZE_EPOCHS = 3
FOCAL_GAMMA = 2.0
BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
DROPOUT_P = 0.30
WEIGHT_DECAY = 2e-4
LABEL_SMOOTHING = 0.10
USE_AMP = True
RUN_TTA_REJECT = False  # K-Fold leggero: abilita solo se vuoi pagare il costo della TTA su tutti i fold

# Reject option cost model (from L:reject_option)
Cc = 1.0   # cost of correct classification
Cr = 3.0   # cost of rejection
Ce = 10.0  # cost of misclassification

DRIVE_ZIP_PATH = '/content/drive/MyDrive/waste_type_identification.zip'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/waste_type_identification'
LOCAL_DATASET_DIR = '/content/waste_type_identification'
RESULTS_DIR = Path(REPO_PATH) / 'results' / RUN_NAME
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/ProjectML_checkpoints') / RUN_NAME

CLASS_NAMES = ['Battery','Clothing','Glass','Metal','Organic','Papery','Plastic','Undifferentiated']
LABEL_MAP = {
    'battery':0, 'clothes':1, 'clothing':1, 'shoes':1,
    'brown':2, 'green':2, 'transparent':2, 'glass':2, 'metal':3, 'organic':4,
    'cardboard':5, 'paper':5, 'papery':5, 'plastic':6, 'undifferentiated':7,
}
IMG_EXTENSIONS = ('.jpg','.jpeg','.png','.bmp','.gif','.tiff','.webp')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## 3. Dataset e split in fold

In [ ]:
local_dataset = Path(LOCAL_DATASET_DIR)
drive_dataset = Path(DRIVE_DATASET_DIR)
drive_zip = Path(DRIVE_ZIP_PATH)
if not local_dataset.exists():
    if drive_dataset.exists():
        shutil.copytree(drive_dataset, local_dataset)
    elif drive_zip.exists():
        local_dataset.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf: zf.extractall(local_dataset)
    else:
        raise FileNotFoundError(f'Non trovo dataset')
print('Dataset:', local_dataset)

In [ ]:
samples = []
for dirpath, _, filenames in os.walk(local_dataset, followlinks=False):
    folder = Path(dirpath).name.lower()
    if not filenames or folder not in LABEL_MAP: continue
    label = LABEL_MAP[folder]
    for fname in filenames:
        if fname.lower().endswith(IMG_EXTENSIONS):
            path = Path(dirpath) / fname
            samples.append({'path':str(path), 'relative_path':str(path.relative_to(local_dataset)),
                            'folder':folder, 'label':label, 'class_name':CLASS_NAMES[label]})

df = pd.DataFrame(samples).sort_values('relative_path').reset_index(drop=True)
print('Totale immagini:', len(df))
print(df['class_name'].value_counts())

# K-Fold cross-validation split
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_assignments = np.zeros(len(df), dtype=int)
for fold_idx, (_, val_positions) in enumerate(skf.split(df['relative_path'], df['label'])):
    fold_assignments[val_positions] = fold_idx
df['fold'] = fold_assignments

split_path = RESULTS_DIR / 'split.csv'
df[['relative_path','folder','label','class_name','fold']].to_csv(split_path, index=False)
print('\nFold distribution:')
for f in range(K_FOLDS):
    fold_df = df[df['fold'] == f]
    print(f'  Fold {f}: {len(fold_df)} samples')
print(pd.crosstab(df['class_name'], df['fold']).reindex(CLASS_NAMES, fill_value=0))

## 4. Transforms e DataLoader per fold

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None: image = self.transform(image)
        return image, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.70, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),
    transforms.ColorJitter(brightness=0.30, contrast=0.25, saturation=0.20, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Pre-build fold dataloaders
fold_train_loaders, fold_val_loaders, fold_val_frames = [], [], []
for f in range(K_FOLDS):
    val_fold_df = df[df['fold'] == f].reset_index(drop=True)
    train_fold_df = df[df['fold'] != f].reset_index(drop=True)

    train_labels = train_fold_df['label'].to_numpy()
    class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    sample_weights = (1.0 / class_counts)[train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    fold_train_loaders.append(DataLoader(
        WasteDataset(train_fold_df, train_transform), batch_size=BATCH_SIZE,
        sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True))
    fold_val_loaders.append(DataLoader(
        WasteDataset(val_fold_df, val_transform), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True))
    fold_val_frames.append(val_fold_df)

print(f'{K_FOLDS}-fold dataloaders ready.')

## 5. Modello (Focal Loss + Two-Phase Training)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight; self.gamma = gamma; self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                            label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def build_model(class_weights_tensor):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Sequential(nn.Dropout(p=DROPOUT_P), nn.Linear(model.fc.in_features, NUM_CLASSES))
    model = model.to(DEVICE)
    criterion = FocalLoss(weight=class_weights_tensor, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': BACKBONE_LR},
        {'params': model.fc.parameters(), 'lr': HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and DEVICE.type == 'cuda')
    return model, criterion, optimizer, scheduler, scaler

def freeze_backbone(model, freeze):
    for name, param in model.named_parameters():
        if not name.startswith('fc.'): param.requires_grad = not freeze

config = {
    'experiment': EXP_NAME, 'run_id': RUN_ID, 'run_name': RUN_NAME,
    'base_branch': BASE_BRANCH, 'run_branch': RUN_BRANCH,
    'model': 'ResNet18 pretrained ImageNet',
    'strategy': 'light_kfold3_StratifiedKFold_two_phase_freeze3ep_15epochs_patience4_FocalLoss_gamma2_LS0.10_RejectOption_Vento_noTTAdefault',
    'seed': SEED, 'k_folds': K_FOLDS, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'patience': PATIENCE, 'two_phase_freeze_epochs': TWO_PHASE_FREEZE_EPOCHS,
    'run_tta_reject': RUN_TTA_REJECT,
    'focal_gamma': FOCAL_GAMMA, 'label_smoothing': LABEL_SMOOTHING,
    'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR,
    'dropout_p': DROPOUT_P, 'weight_decay': WEIGHT_DECAY,
    'reject_costs': {'Cc': Cc, 'Cr': Cr, 'Ce': Ce},
    'reliability_functions': ['psi_a (max softmax)', 'psi_b (top2 margin)'],
    'course_arguments': 'transfer learning (S:12), two-phase (S:12), discriminative LRs (S:12), WeightedRandomSampler (S:05), label smoothing (S:09), focal loss (S:09), ReduceLROnPlateau (S:09), early stopping (S:09), dropout (S:13), data augmentation (S:05), K-Fold CV (S:04, L:01), Reject Option Vento (S:11, L:reject_option), balanced accuracy (S:05); TTA opzionale ma disattivata per contenere il costo computazionale',
    'results_dir': str(RESULTS_DIR),
}
with open(RESULTS_DIR / 'config.json', 'w') as f: json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 6. Training K-Fold

In [ ]:
def run_epoch(model, loader, train, criterion, optimizer, scaler):
    model.train(train)
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, targets in loader:
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
            if train: optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP and DEVICE.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, targets)
            if train:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * images.size(0)
            all_preds.append(logits.argmax(dim=1).detach().cpu())
            all_targets.append(targets.detach().cpu())
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_targets).numpy()
    return total_loss / len(loader.dataset), float((y_pred == y_true).mean()), float(balanced_accuracy_score(y_true, y_pred)), y_true, y_pred

def train_one_fold(fold_idx, train_loader, val_loader, train_df):
    train_counts = train_df['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(np.float32)
    cw = torch.tensor(np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0)), dtype=torch.float32, device=DEVICE)
    cw = cw / cw.mean()

    model, criterion, optimizer, scheduler, scaler = build_model(cw)
    ckpt_path = DRIVE_CHECKPOINT_DIR / f'fold{fold_idx}_best.pth'

    best_ba, best_ep, no_impr, history = -1.0, -1, 0, []

    # Phase 1: freeze backbone
    freeze_backbone(model, True)
    print(f'Fold {fold_idx} Phase 1: backbone frozen ({TWO_PHASE_FREEZE_EPOCHS} epochs)')
    for ep in range(1, TWO_PHASE_FREEZE_EPOCHS + 1):
        tl, ta, tba, _, _ = run_epoch(model, train_loader, True, criterion, optimizer, scaler)
        vl, va, vba, _, _ = run_epoch(model, val_loader, False, criterion, optimizer, scaler)
        history.append({'epoch':ep, 'phase':'freeze', 'fold':fold_idx, 'train_loss':tl, 'train_ba':tba, 'val_loss':vl, 'val_ba':vba})
        print(f'  E{ep:02d} [freeze]: tl={tl:.3f} tba={tba:.4f} vl={vl:.3f} vba={vba:.4f}')
        if vba > best_ba:
            best_ba, best_ep = vba, ep
            torch.save({'model_state_dict': model.state_dict(), 'fold': fold_idx, 'best_epoch': ep, 'best_val_ba': best_ba}, ckpt_path)
            print(f'    Saved best: BA={best_ba:.4f}')
        scheduler.step(vba)

    # Phase 2: unfreeze
    freeze_backbone(model, False)
    no_impr = 0
    print(f'Fold {fold_idx} Phase 2: backbone UNFROZEN')
    for ep in range(TWO_PHASE_FREEZE_EPOCHS + 1, EPOCHS + 1):
        tl, ta, tba, _, _ = run_epoch(model, train_loader, True, criterion, optimizer, scaler)
        vl, va, vba, yt, yp = run_epoch(model, val_loader, False, criterion, optimizer, scaler)
        history.append({'epoch':ep, 'phase':'finetune', 'fold':fold_idx, 'train_loss':tl, 'train_ba':tba, 'val_loss':vl, 'val_ba':vba})
        print(f'  E{ep:02d} [finetune]: tl={tl:.3f} tba={tba:.4f} vl={vl:.3f} vba={vba:.4f}')
        if vba > best_ba:
            best_ba, best_ep, no_impr = vba, ep, 0
            torch.save({'model_state_dict': model.state_dict(), 'fold': fold_idx, 'best_epoch': ep, 'best_val_ba': best_ba}, ckpt_path)
            print(f'    Saved best: BA={best_ba:.4f}')
        else:
            no_impr += 1
            if no_impr >= PATIENCE:
                print(f'    Early stop at epoch {ep}')
                break
        scheduler.step(vba)

    pd.DataFrame(history).to_csv(RESULTS_DIR / f'history_fold{fold_idx}.csv', index=False)
    return ckpt_path, best_ba, best_ep, model, history

fold_results = []
for f in range(K_FOLDS):
    print(f'\n===== Fold {f}/{K_FOLDS} =====')
    train_df_fold = df[df['fold'] != f]
    ckpt, ba, ep, model, hist = train_one_fold(f, fold_train_loaders[f], fold_val_loaders[f], train_df_fold)
    fold_results.append({'fold': f, 'checkpoint': str(ckpt), 'best_ba': ba, 'best_epoch': ep})
    print(f'Fold {f} best BA: {ba:.4f} at epoch {ep}')

pd.DataFrame(fold_results).to_csv(RESULTS_DIR / 'fold_summary.csv', index=False)
bas = [r['best_ba'] for r in fold_results]
print(f'\nK-Fold BA: mean={np.mean(bas):.4f} +/- {np.std(bas):.4f}')
print(f'  Individual folds: {[f"{b:.4f}" for b in bas]}')

## 7. Reject Option (Vento et al., S:11)

In [ ]:
def compute_psi_and_reject(model, val_loader, val_df):
    """Compute softmax probabilities and both psi values for entire validation set."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(targets.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    psi_a = sorted_probs[:, 0]                    # max class probability
    psi_b = sorted_probs[:, 0] - sorted_probs[:, 1]  # top-2 margin

    return psi_a, psi_b, probs, labels

def find_optimal_threshold(psi, labels, probs, Cc=1, Cr=3, Ce=10):
    """Find optimal reject threshold using effectiveness function P (Vento et al.)."""
    preds = probs.argmax(axis=1)
    correct = (preds == labels)
    error = ~correct
    N = len(labels)
    total_correct = correct.sum()
    total_error = error.sum()

    candidate_thresholds = np.unique(psi)
    best_P, best_tau, best_metrics = -np.inf, 0.0, None

    for tau in candidate_thresholds:
        rejected = psi < tau
        accepted = ~rejected
        if accepted.sum() == 0: continue

        Rc = correct[accepted].sum() / N
        Re = error[accepted].sum() / N
        Rr = rejected.sum() / N
        Rc0 = total_correct / N
        Re0 = total_error / N

        P = Cc * (Rc - Rc0) - Ce * (Re - Re0) - Cr * Rr
        if P > best_P:
            best_P = P
            best_tau = tau
            best_metrics = {
                'threshold': float(tau), 'P_effectiveness': float(P),
                'reject_rate': float(Rr),
                'coverage': float(1 - Rr),
                'accepted_accuracy': float(correct[accepted].sum() / accepted.sum()),
                'accepted_balanced_accuracy': float(balanced_accuracy_score(labels[accepted], preds[accepted])),
                'n_accepted': int(accepted.sum()), 'n_rejected': int(rejected.sum()),
                'errors_avoided': int(error[rejected].sum()),
            }
        elif tau == 1.0 and best_tau == 0.0:
            best_tau = tau; best_metrics = best_metrics or {'threshold': 1.0, 'reject_rate': 0.0, 'coverage': 1.0}

    return best_tau, best_metrics

reject_results = []
for f in range(K_FOLDS):
    print(f'\n--- Fold {f}: Reject Option ---')
    ckpt = torch.load(fold_results[f]['checkpoint'], map_location=DEVICE, weights_only=False)
    train_df_fold = df[df['fold'] != f]
    train_counts = train_df_fold['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(np.float32)
    cw = torch.tensor(np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0)), dtype=torch.float32, device=DEVICE)
    cw = cw / cw.mean()
    model, _, _, _, _ = build_model(cw)
    model.load_state_dict(ckpt['model_state_dict'])

    psi_a, psi_b, probs, labels = compute_psi_and_reject(model, fold_val_loaders[f], fold_val_frames[f])

    for psi_name, psi_vals in [('psi_a', psi_a), ('psi_b', psi_b)]:
        tau, metrics = find_optimal_threshold(psi_vals, labels, probs, Cc, Cr, Ce)
        metrics['fold'] = f
        metrics['psi_type'] = psi_name
        reject_results.append(metrics)
        print(f"  {psi_name}: tau={tau:.3f} | reject={metrics['reject_rate']:.1%} | accepted_BA={metrics['accepted_balanced_accuracy']:.4f} | errors_avoided={metrics['errors_avoided']}")

reject_df = pd.DataFrame(reject_results)
reject_df.to_csv(RESULTS_DIR / 'reject_results.csv', index=False)

print('\n===== Reject Option Summary =====')
for psi_type in ['psi_a', 'psi_b']:
    sub = reject_df[reject_df['psi_type'] == psi_type]
    print(f"\n{psi_type} (mean +/- std across {K_FOLDS} folds):")
    print(f"  Threshold:        {sub['threshold'].mean():.4f} +/- {sub['threshold'].std():.4f}")
    print(f"  Reject rate:      {sub['reject_rate'].mean():.2%} +/- {sub['reject_rate'].std():.2%}")
    print(f"  Accepted BA:      {sub['accepted_balanced_accuracy'].mean():.4f} +/- {sub['accepted_balanced_accuracy'].std():.4f}")
    print(f"  Accepted Acc:     {sub['accepted_accuracy'].mean():.4f} +/- {sub['accepted_accuracy'].std():.4f}")
    print(f"  Errors avoided:   {sub['errors_avoided'].mean():.0f} +/- {sub['errors_avoided'].std():.0f}")

best_psi = 'psi_a' if reject_df[reject_df['psi_type']=='psi_a']['accepted_balanced_accuracy'].mean() > reject_df[reject_df['psi_type']=='psi_b']['accepted_balanced_accuracy'].mean() else 'psi_b'
print(f'\nBest reliability function: {best_psi}')

## 8. TTA 10-crop con Reject Option (sul best model per fold)

In [ ]:
crop_224 = transforms.FiveCrop(224)
tta_normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
to_tensor = transforms.ToTensor()

def tta_predict_probs(model, image):
    """10-crop TTA: returns average softmax probabilities."""
    image_256 = transforms.Resize(256)(image)
    crops = crop_224(image_256)
    all_probs = []
    for crop in crops:
        for flip in [False, True]:
            c = transforms.functional.hflip(crop) if flip else crop
            t = tta_normalize(to_tensor(c)).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                all_probs.append(F.softmax(model(t), dim=1))
    return torch.stack(all_probs).mean(dim=0).cpu().numpy()

def tta_evaluate_with_psi(model, val_df):
    """TTA 10-crop on full validation set, returning psi_a, psi_b, probs, labels."""
    model.eval()
    all_probs, all_labels = [], []
    for idx in range(len(val_df)):
        row = val_df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        probs = tta_predict_probs(model, image)
        all_probs.append(probs.flatten())
        all_labels.append(int(row['label']))
        if (idx + 1) % 500 == 0: print(f'  TTA: {idx+1}/{len(val_df)}')
    probs = np.array(all_probs)
    labels = np.array(all_labels)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    psi_a = sorted_probs[:, 0]
    psi_b = sorted_probs[:, 0] - sorted_probs[:, 1]
    return psi_a, psi_b, probs, labels

tta_reject_results = []
if RUN_TTA_REJECT:
    print('\n===== TTA 10-crop + Reject Option =====')
else:
    print('\n===== TTA 10-crop + Reject Option SKIPPED (RUN_TTA_REJECT=False) =====')
for f in range(K_FOLDS) if RUN_TTA_REJECT else []:
    print(f'\n--- Fold {f}: TTA + Reject ---')
    ckpt = torch.load(fold_results[f]['checkpoint'], map_location=DEVICE, weights_only=False)
    train_df_fold = df[df['fold'] != f]
    train_counts = train_df_fold['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(np.float32)
    cw = torch.tensor(np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0)), dtype=torch.float32, device=DEVICE)
    cw = cw / cw.mean()
    model, _, _, _, _ = build_model(cw)
    model.load_state_dict(ckpt['model_state_dict'])

    psi_a, psi_b, probs, labels = tta_evaluate_with_psi(model, fold_val_frames[f])

    for psi_name, psi_vals in [('psi_a', psi_a), ('psi_b', psi_b)]:
        tau, metrics = find_optimal_threshold(psi_vals, labels, probs, Cc, Cr, Ce)
        metrics['fold'] = f; metrics['psi_type'] = psi_name
        tta_reject_results.append(metrics)
        print(f"  TTA {psi_name}: tau={tau:.3f} | reject={metrics['reject_rate']:.1%} | BA={metrics['accepted_balanced_accuracy']:.4f}")

tta_reject_df = pd.DataFrame(tta_reject_results)

if RUN_TTA_REJECT:
    print('\n--- TTA Reject Option Summary ---')
    for psi_type in ['psi_a', 'psi_b']:
        sub = tta_reject_df[tta_reject_df['psi_type'] == psi_type]
        print(f"\nTTA {psi_type}:")
        print(f"  Reject rate: {sub['reject_rate'].mean():.2%} +/- {sub['reject_rate'].std():.2%}")
        print(f"  Accepted BA: {sub['accepted_balanced_accuracy'].mean():.4f} +/- {sub['accepted_balanced_accuracy'].std():.4f}")
else:
    tta_reject_df = pd.DataFrame(columns=['fold','psi_type','accepted_balanced_accuracy','accepted_accuracy','reject_rate','coverage'])
tta_reject_df.to_csv(RESULTS_DIR / 'tta_reject_results.csv', index=False)

## 9. Riepilogo finale, grafici e report

In [ ]:
print('\n========== FINAL SUMMARY v8 ==========')
print(f'K-Fold BA: {np.mean(bas):.4f} +/- {np.std(bas):.4f}')

print('\n--- Standard Reject Option ---')
for psi in ['psi_a', 'psi_b']:
    s = reject_df[reject_df['psi_type'] == psi]
    print(f'{psi}: BA={s["accepted_balanced_accuracy"].mean():.4f}+/-{s["accepted_balanced_accuracy"].std():.4f}  reject={s["reject_rate"].mean():.1%}  coverage={s["coverage"].mean():.1%}')

if RUN_TTA_REJECT:
    print('\n--- TTA Reject Option ---')
    for psi in ['psi_a', 'psi_b']:
        s = tta_reject_df[tta_reject_df['psi_type'] == psi]
        print(f'TTA {psi}: BA={s["accepted_balanced_accuracy"].mean():.4f}+/-{s["accepted_balanced_accuracy"].std():.4f}  reject={s["reject_rate"].mean():.1%}  coverage={s["coverage"].mean():.1%}')
else:
    print('\n--- TTA Reject Option skipped: RUN_TTA_REJECT=False ---')

# Save master summary
summary = {
    'run_name': RUN_NAME, 'run_branch': RUN_BRANCH,
    'k_folds': K_FOLDS, 'epochs': EPOCHS, 'patience': PATIENCE,
    'two_phase_freeze_epochs': TWO_PHASE_FREEZE_EPOCHS,
    'run_tta_reject': RUN_TTA_REJECT,
    'kfold_ba_mean': float(np.mean(bas)), 'kfold_ba_std': float(np.std(bas)),
    'fold_bas': [float(b) for b in bas],
    'standard_psi_a': {
        'ba_mean': float(reject_df[reject_df['psi_type']=='psi_a']['accepted_balanced_accuracy'].mean()),
        'ba_std': float(reject_df[reject_df['psi_type']=='psi_a']['accepted_balanced_accuracy'].std()),
        'reject_rate_mean': float(reject_df[reject_df['psi_type']=='psi_a']['reject_rate'].mean()),
    },
    'standard_psi_b': {
        'ba_mean': float(reject_df[reject_df['psi_type']=='psi_b']['accepted_balanced_accuracy'].mean()),
        'ba_std': float(reject_df[reject_df['psi_type']=='psi_b']['accepted_balanced_accuracy'].std()),
        'reject_rate_mean': float(reject_df[reject_df['psi_type']=='psi_b']['reject_rate'].mean()),
    },
    'tta_psi_a': ({
        'ba_mean': float(tta_reject_df[tta_reject_df['psi_type']=='psi_a']['accepted_balanced_accuracy'].mean()),
        'ba_std': float(tta_reject_df[tta_reject_df['psi_type']=='psi_a']['accepted_balanced_accuracy'].std()),
        'reject_rate_mean': float(tta_reject_df[tta_reject_df['psi_type']=='psi_a']['reject_rate'].mean()),
    } if RUN_TTA_REJECT else None),
    'tta_psi_b': ({
        'ba_mean': float(tta_reject_df[tta_reject_df['psi_type']=='psi_b']['accepted_balanced_accuracy'].mean()),
        'ba_std': float(tta_reject_df[tta_reject_df['psi_type']=='psi_b']['accepted_balanced_accuracy'].std()),
        'reject_rate_mean': float(tta_reject_df[tta_reject_df['psi_type']=='psi_b']['reject_rate'].mean()),
    } if RUN_TTA_REJECT else None),
    'best_reliability': best_psi,
}
with open(RESULTS_DIR / 'summary.json', 'w') as f: json.dump(summary, f, indent=2)
print('\nSummary saved to summary.json')

# Histories plot (fold 0 only for brevity)
try:
    h0 = pd.read_csv(RESULTS_DIR / 'history_fold0.csv')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for phase, color in [('freeze', 'orange'), ('finetune', 'blue')]:
        ph = h0[h0['phase'] == phase]
        if len(ph) > 0:
            axes[0].plot(ph['epoch'], ph['train_loss'], color=color, alpha=0.5, label=f'train {phase}')
            axes[0].plot(ph['epoch'], ph['val_loss'], color=color, linestyle='--', label=f'val {phase}')
            axes[1].plot(ph['epoch'], ph['train_ba'], color=color, alpha=0.5, label=f'train {phase}')
            axes[1].plot(ph['epoch'], ph['val_ba'], color=color, linestyle='--', label=f'val {phase}')
    axes[0].set_title('v8 Fold 0 Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].set_title('v8 Fold 0 BA'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout(); plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=160); plt.show()
except: pass

print('\nDone.')

## 10. Commit e push

In [ ]:
os.chdir(REPO_PATH)
run(['git', 'config', 'user.name', 'fabriziodrr'], cwd=REPO_PATH)
run(['git', 'config', 'user.email', 'fabriziodrr@users.noreply.github.com'], cwd=REPO_PATH)
run(['git', 'add', str(RESULTS_DIR.relative_to(REPO_PATH))], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()
if not status:
    print('Nessuna modifica.')
else:
    run(['git', 'commit', '-m', f'Add ResNet18 exnovo v8 light kfold results {RUN_ID}'], cwd=REPO_PATH)
    auth_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'push', '-u', 'origin', RUN_BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)
print('Done.')